# Project 4 Workbook: Responsible AI Fairness Monitoring in Production

You are the external responsible-AI and model-risk audit team for a mortgage decision-support system.

Your goal is not only to train a model. Your goal is to decide whether the system should be deployed as-is, deployed with controls, returned for remediation, or rejected.

Primary dataset: prepared HMDA project slice in `data/hmda_project4_slice.csv.gz`.


## Submission Rules (Read First)

This workbook is graded for responsible-AI reasoning, not for maximizing one model metric.

Required outputs:
1. M0 data and model-risk assessment.
2. M1 baseline models, validation discipline, calibration, and threshold policy.
3. M2 fairness audit across multiple definitions.
4. M3 at least two mitigation approaches from different categories.
5. M4 production monitoring simulation with at least one fairness incident investigation.
6. M5 final deployment recommendation and model card.

Coding expectation:
- Cells marked `TODO REQUIRED` must be implemented by students.
- You may reuse ideas from the tutorial, but you must write your own HMDA-specific implementation.
- Do not leave placeholder text in required reflection sections.

Not allowed:
- Treating historical HMDA labels as neutral ground truth.
- Claiming fairness from one metric only.
- Dropping protected attributes and declaring the model fair.
- Tuning thresholds or mitigation choices on the test set.
- Reporting fairness metrics without sample sizes.


## Milestone Map

- M0: Data and Model Risk Assessment
- M1: Baseline Models and Threshold Policy
- M2: Fairness Audit Across Definitions
- M3: Mitigation and Trade-Off Analysis
- M4: Production Monitoring and Incident Response
- M5: Deployment Recommendation and Model Card


## Anti-Shortcut Protocol (Mandatory)

Your reasoning must be auditable.

Required controls:
1. Evidence trace tags for every major claim.
2. A fairness-definition decision log.
3. A mitigation trade-off log.
4. An incident investigation worksheet.
5. A final evidence map.
6. Oral-defense readiness notes.

Use tags such as:
- `[D1]` data/risk evidence
- `[M1]` model evidence
- `[F1]` fairness evidence
- `[I1]` individual/proxy evidence
- `[G1]` mitigation evidence
- `[P1]` production monitoring evidence
- `[R1]` recommendation evidence


## Team Configuration (Fill This)

- Team ID:
- Team members:
- HMDA slice file used:
- Years covered:
- State/region covered:
- Prediction target definition:
- Positive outcome definition:
- Negative outcome definition:
- Ambiguous/excluded outcome note:
- Primary deployment decision supported by model:
- Operational capacity assumption:
- AI/source disclosure:


## Minimal Setup

This cell only creates common paths and imports. You are responsible for the actual pipeline.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

RANDOM_SEED = 4146
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)


## M0: Data and Model Risk Assessment

Minimum requirements:
1. Load the HMDA slice and inspect schema, row count, and missingness.
2. Define the decision context and affected population.
3. Construct the target label from HMDA action outcomes.
4. Verify the prepared slice outcome mapping and explain that withdrawn, incomplete, purchased, preapproval, or otherwise ambiguous records are excluded from this modeling slice. If you create any raw-data extension, handle those records separately.
5. Identify protected and vulnerable groups available in the data.
6. Report group sample sizes and missing demographic values.
7. Discuss why historical action outcomes are not objective ground truth.

Required evidence:
- Dataset shape and target distribution `[D1]`
- Outcome mapping table `[D2]`
- Protected-group sample-size table `[D3]`
- Historical-label risk paragraph `[D4]`


In [37]:
# TODO REQUIRED: Load the HMDA project slice.
# Example:
DATA_FILE = DATA_DIR / "hmda_project4_slice.csv.gz"
df = pd.read_csv(DATA_FILE)
print(df.head(3))


   activity_year state_code               action_taken  loan_type  loan_purpose  lien_status  occupancy_type  loan_amount  income  \
0           2022         CA  Approved but not accepted          1             1            1               1    1505000.0   439.0   
1           2022         CA                 Originated          2             1            1               1     425000.0    38.0   
2           2022         CA                 Originated          2             1            1               1     325000.0    22.0   

  debt_to_income_ratio  loan_to_value_ratio  property_value  interest_rate  rate_spread  total_loan_costs age_group race_group  \
0                  NaN                  NaN             NaN            NaN          NaN               NaN     45-54      Asian   
1                  NaN                  NaN        655000.0           1.64          NaN               NaN       >74      White   
2                  NaN                  NaN        555000.0           3.06   

In [26]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 850278 entries, 0 to 850277
Data columns (total 28 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   activity_year                      850278 non-null  int64  
 1   state_code                         850278 non-null  str    
 2   action_taken                       850278 non-null  str    
 3   loan_type                          850278 non-null  int64  
 4   loan_purpose                       850278 non-null  int64  
 5   lien_status                        850278 non-null  int64  
 6   occupancy_type                     850278 non-null  int64  
 7   loan_amount                        850278 non-null  float64
 8   income                             840157 non-null  float64
 9   debt_to_income_ratio               835674 non-null  str    
 10  loan_to_value_ratio                797934 non-null  float64
 11  property_value                     837121 non-null

In [27]:
print(df.describe(include="all"))

        activity_year state_code action_taken      loan_type  loan_purpose  lien_status  occupancy_type   loan_amount        income  \
count   850278.000000     850278       850278  850278.000000      850278.0     850278.0        850278.0  8.502780e+05  8.401570e+05   
unique            NaN          1            3            NaN           NaN          NaN             NaN           NaN           NaN   
top               NaN         CA   Originated            NaN           NaN          NaN             NaN           NaN           NaN   
freq              NaN     850278       720578            NaN           NaN          NaN             NaN           NaN           NaN   
mean      2022.904137        NaN          NaN       1.296180           1.0          1.0             1.0  6.604959e+05  2.257056e+02   
std          0.834638        NaN          NaN       0.587512           0.0          0.0             0.0  5.459007e+05  9.042843e+03   
min       2022.000000        NaN          NaN       1.0

In [28]:
print(df.shape)
print(df.columns)

(850278, 28)
Index(['activity_year', 'state_code', 'action_taken', 'loan_type', 'loan_purpose', 'lien_status', 'occupancy_type', 'loan_amount',
       'income', 'debt_to_income_ratio', 'loan_to_value_ratio', 'property_value', 'interest_rate', 'rate_spread', 'total_loan_costs',
       'age_group', 'race_group', 'ethnicity_group', 'sex_group', 'ffiec_msa_md_median_family_income', 'tract_minority_share',
       'tract_income_ratio', 'tract_population', 'county_code', 'census_tract', 'lei', 'outcome_group', 'target_denied'],
      dtype='str')


In [29]:
print(df.isnull().sum())

activity_year                             0
state_code                                0
action_taken                              0
loan_type                                 0
loan_purpose                              0
lien_status                               0
occupancy_type                            0
loan_amount                               0
income                                10121
debt_to_income_ratio                  14604
loan_to_value_ratio                   52344
property_value                        13157
interest_rate                         98974
rate_spread                          111398
total_loan_costs                     163883
age_group                                 0
race_group                                0
ethnicity_group                           0
sex_group                                 0
ffiec_msa_md_median_family_income         0
tract_minority_share                      0
tract_income_ratio                        0
tract_population                

In [36]:
print(df.dtypes)

activity_year                          int64
state_code                               str
action_taken                             str
loan_type                              int64
loan_purpose                           int64
lien_status                            int64
occupancy_type                         int64
loan_amount                          float64
income                               float64
debt_to_income_ratio                     str
loan_to_value_ratio                  float64
property_value                       float64
interest_rate                        float64
rate_spread                          float64
total_loan_costs                     float64
age_group                                str
race_group                               str
ethnicity_group                          str
sex_group                                str
ffiec_msa_md_median_family_income      int64
tract_minority_share                 float64
tract_income_ratio                   float64
tract_popu

In [31]:
print(df.nunique())

activity_year                             3
state_code                                1
action_taken                              3
loan_type                                 4
loan_purpose                              1
lien_status                               1
occupancy_type                            1
loan_amount                             860
income                                 4043
debt_to_income_ratio                     20
loan_to_value_ratio                   57944
property_value                         1158
interest_rate                          5608
rate_spread                           30298
total_loan_costs                     486182
age_group                                 8
race_group                                9
ethnicity_group                           5
sex_group                                 4
ffiec_msa_md_median_family_income        82
tract_minority_share                   5395
tract_income_ratio                     6696
tract_population                

In [34]:
print(df.columns[df.nunique() == 1])

Index(['state_code', 'loan_purpose', 'lien_status', 'occupancy_type'], dtype='str')


In [41]:
df['target_denied'].value_counts(normalize=True)

target_denied
0    0.888146
1    0.111854
Name: proportion, dtype: float64

In [45]:
df.groupby('target_denied')['interest_rate'].apply(lambda s: s.isna().mean())

for col in ['interest_rate', 'rate_spread', 'total_loan_costs', 'loan_to_value_ratio', 'property_value']:
    print(col)
    print(df.groupby('target_denied')[col].apply(lambda s: s.isna().mean()))

interest_rate
target_denied
0    0.005121
1    1.000000
Name: interest_rate, dtype: float64
rate_spread
target_denied
0    0.021573
1    1.000000
Name: rate_spread, dtype: float64
total_loan_costs
target_denied
0    0.091073
1    1.000000
Name: total_loan_costs, dtype: float64
loan_to_value_ratio
target_denied
0    0.043452
1    0.205348
Name: loan_to_value_ratio, dtype: float64
property_value
target_denied
0    0.012957
1    0.035455
Name: property_value, dtype: float64


Loan-to-Value is higher for denied applications (20% vs 4%) that means the target_denied = 1 for 20% of the blanks is nothing just small as compared to the ones who are approved 4%.

In [47]:
# TODO REQUIRED: Verify and justify the target label.
# Include a printed mapping from HMDA action/outcome values in the prepared slice to your modeling label.
# State which ambiguous outcomes are absent from the prepared slice and why that matters.

mapping_table = df.groupby(['action_taken', 'outcome_group', 'target_denied']).size()
print(mapping_table)

import json
with open('data/hmda_project4_slice_metadata.json') as f:
    meta = json.load(f)
print(meta['filters'])

action_taken               outcome_group           target_denied
Approved but not accepted  approved_or_originated  0                 34593
Denied                     denied                  1                 95107
Originated                 approved_or_originated  0                720578
dtype: int64
{'actions_taken': '1,2,3', 'loan_purposes': '1', 'local_post_filter_lien_status': '1 when column is present', 'local_post_filter_occupancy_type': '1 when column is present'}


[D2] Outcome mapping. This slice maps action_taken values Originated and Approved but not accepted to target_denied = 0, and Denied to target_denied = 1. We treat these two "approved" categories the same because the bank approves "originated" and "appreoved but not accepted" and what we are trying to figure out is whether the bank said yes or no which means in these both cases the bank said yes.

This slice was pulled using actions_taken=1,2,3 only (per the metadata filters) and according to https://www.ffiec.gov/sites/default/files/data/hmda/code.pdf, we are missing 4, 5, 6:
1 - Originated (loan approved)
2 - Application approved but not accepted (maybe because they got a better offer or something)
3 - Denied
4 - Withdrawn
5 - Closed for incompleteness
6 - Purchased Loan (this is like a different bank took someone else's already finished loan)


This matters for the audit because why should be consider people who withdrew or closed for incomplete informationn and purchased loan. It does not matter in the end whether the bank is accepting or not.

In [49]:
for col in ['race_group', 'ethnicity_group', 'sex_group']:
    print(df[col].value_counts())
    print()

race_group
White                                        416514
Race Not Available                           192395
Asian                                        164561
Joint                                         34397
Black or African American                     31014
American Indian or Alaska Native               6033
Native Hawaiian or Other Pacific Islander      2738
2 or more minority races                       2390
Free Form Text Only                             236
Name: count, dtype: int64

ethnicity_group
Not Hispanic or Latino     460442
Hispanic or Latino         196463
Ethnicity Not Available    153970
Joint                       38717
Free Form Text Only           686
Name: count, dtype: int64

sex_group
Joint                366671
Male                 259815
Female               168534
Sex Not Available     55258
Name: count, dtype: int64



In [50]:
for col in ['age_group']:
    print(df[col].value_counts())
    print()

age_group
35-44    268188
25-34    263492
45-54    147483
55-64     89262
65-74     41126
<25       26929
>74       13162
8888        636
Name: count, dtype: int64



In [53]:
df.groupby(['race_group','sex_group']).size().sort_values()

race_group                                 sex_group        
Free Form Text Only                        Sex Not Available        20
2 or more minority races                   Sex Not Available        26
Native Hawaiian or Other Pacific Islander  Sex Not Available        26
Free Form Text Only                        Female                   41
American Indian or Alaska Native           Sex Not Available        54
Free Form Text Only                        Male                     87
                                           Joint                    88
Joint                                      Sex Not Available       120
Black or African American                  Sex Not Available       201
Native Hawaiian or Other Pacific Islander  Female                  678
2 or more minority races                   Female                  737
                                           Joint                   764
                                           Male                    863
Native Hawaiian 

### M0 Reflection (Write Here)

Use this structure:
1. Who is affected by this model-assisted decision?
2. Which outcome records are absent from the prepared slice, and why should they not be treated as approved or denied? If you changed the slice, what did you exclude or separate?
3. Which groups have enough sample size for stable audit claims?
4. What are the biggest historical-bias risks in the label?


- This is not just mortage applicant but maybe co-applicants too or communities where if get denied can affect in certain ways.

- the action_taken is missing 4, 5, 6 and they should not be treated as it will affect the model when training or simply put, we are auditing based on the bank decision and the applications getting withdrawn, purchased loans, or closed by incompleteness are some factors that the bannk has no importance on.

- 

## M1: Baseline Models and Threshold Policy

Minimum requirements:
1. Create leakage-safe features available at decision time.
2. Split data into training, validation, and test sets. If time fields exist, use a time-aware split or justify another policy.
3. Train at least two models: one interpretable baseline and one higher-capacity model.
4. Report ROC-AUC or PR-AUC, precision, recall, F1, confusion matrix, and calibration evidence.
5. Choose an operational threshold on validation data only.
6. Explain the business objective optimized by your threshold.

Required evidence:
- Feature availability/leakage table `[M1]`
- Split summary `[M2]`
- Model comparison table `[M3]`
- Calibration evidence `[M4]`
- Threshold/capacity trade-off table `[M5]`


In [4]:
# TODO REQUIRED: Build feature matrix X and label y.
# Document which fields are excluded because they leak outcomes or post-decision information.

# Your code here


In [5]:
# TODO REQUIRED: Create train/validation/test split.
# Keep test data untouched until the final evaluation.

# Your code here


In [6]:
# TODO REQUIRED: Train at least two baseline models and evaluate validation performance.

# Your code here


In [7]:
# TODO REQUIRED: Evaluate calibration and choose an operational threshold on validation data.

# Your code here


### M1 Reflection (Write Here)

Use this structure:
1. Which model did you select for the main audit and why?
2. What threshold did you select, and what trade-off did it accept?
3. Which error type is more harmful in this decision context?
4. What model limitation matters most before deployment?


## M2: Fairness Audit Across Definitions

You must evaluate more than one fairness definition and explain trade-offs among them.

Minimum group/intersectional requirements:
1. Analyze sex, race/ethnicity, age, and at least two intersectional groups.
2. Include sample sizes for every group table.
3. Report at least three group fairness families, such as demographic parity, equal opportunity, equalized odds, predictive parity, calibration within groups, and worst-group performance.
4. Identify where fairness definitions disagree.
5. State which fairness definition is most relevant to your deployment decision and why.

Minimum individual/proxy requirements:
1. Define a similarity function using non-sensitive, decision-relevant features.
2. Compute an individual fairness diagnostic, such as nearest-neighbor score inconsistency.
3. Perform proxy analysis by trying to reconstruct at least one protected attribute from non-protected features.
4. Explain why removing protected attributes alone is insufficient.

Required evidence:
- Group fairness tables `[F1]`
- Intersectional fairness tables `[F2]`
- Fairness-definition trade-off table `[F3]`
- Individual fairness diagnostic `[I1]`
- Proxy reconstruction analysis `[I2]`


In [8]:
# TODO REQUIRED: Compute group fairness metrics.
# Include n, base rate, selection/approval rate, TPR, FPR, FNR, precision/PPV, and uncertainty notes.

# Your code here


In [9]:
# TODO REQUIRED: Compute at least two intersectional fairness analyses.
# Examples: race_ethnicity x sex, age_group x race_ethnicity, sex x loan_purpose.

# Your code here


In [10]:
# TODO REQUIRED: Compare fairness definitions across thresholds or model variants.
# Show at least one trade-off where improving one fairness metric worsens another metric or utility metric.

# Your code here


In [11]:
# TODO REQUIRED: Implement individual fairness diagnostic and proxy reconstruction analysis.

# Your code here


### Fairness-Definition Decision Log (Mandatory)

Fill this table in prose or markdown.

| Fairness definition | What it protects against | Evidence tag | What it misses | Decision relevance |
|---|---|---|---|---|
| Demographic parity |  |  |  |  |
| Equal opportunity / equalized odds |  |  |  |  |
| Predictive parity / calibration |  |  |  |  |
| Individual similarity / proxy risk |  |  |  |  |

Conclude with: Which definition receives the most weight in your deployment recommendation, and why?


## M3: Mitigation and Trade-Off Analysis

Implement at least two mitigation approaches from different categories.

Allowed categories:
1. Data or feature intervention: reweighting, resampling, proxy feature removal, feature transformation.
2. Training intervention: fairness-aware objective, constrained model, group-robust or cost-sensitive learning.
3. Decision intervention: threshold adjustment, reject-option review, calibrated post-processing, human-review routing.

Minimum requirements:
1. Compare baseline vs each mitigation using utility and fairness metrics.
2. Explain which groups benefit and which do not.
3. Report costs, calibration changes, and any harmed-group trade-offs.
4. Do not declare success from one improved fairness metric alone.

Required evidence:
- Mitigation design table `[G1]`
- Baseline vs mitigation comparison `[G2]`
- Harmed/benefited group analysis `[G3]`
- Mitigation recommendation with limitations `[G4]`


In [12]:
# TODO REQUIRED: Implement mitigation approach 1.

# Your code here


In [13]:
# TODO REQUIRED: Implement mitigation approach 2 from a different category.

# Your code here


In [14]:
# TODO REQUIRED: Compare baseline and mitigated systems on utility, calibration, and multiple fairness definitions.

# Your code here


### Mitigation Trade-Off Log (Mandatory)

For each mitigation, answer:
1. What changed technically?
2. Which fairness metric improved?
3. Which utility or fairness metric worsened?
4. Which groups benefited?
5. Which groups remained at risk?
6. Would you recommend this mitigation for deployment? Why or why not?


## M4: Production Monitoring and Incident Response

Minimum requirements:
1. Create production-era batches using time, geography, lender, or another justified ordering.
2. Track data quality, feature/score drift, group composition, performance by group, and fairness by group/intersection.
3. Define alert thresholds and minimum sample-size rules.
4. Trigger at least one simulated alert.
5. Complete an investigation worksheet for that alert.

Required evidence:
- Batch construction and reference period `[P1]`
- Monitoring metric table `[P2]`
- Alert policy `[P3]`
- Incident investigation worksheet `[P4]`


In [15]:
# TODO REQUIRED: Build production-style batches and monitoring tables.

# Your code here


In [16]:
# TODO REQUIRED: Define alert rules and trigger at least one simulated incident.

# Your code here


### Incident Investigation Worksheet (Mandatory)

For one alert, complete:

- Alert ID:
- Affected population:
- Metric that changed:
- Reference value:
- Alert-period value:
- When the change began:
- Minimum sample-size check:
- Candidate causes:
- Supporting analyses:
- Immediate containment action:
- Long-term remediation:
- Conditions for returning to normal operation:
- Why this alert does or does not change your deployment recommendation:


## M5: Final Recommendation and Model Card

Your final decision must be one of:
1. Deploy as-is.
2. Deploy with controls.
3. Return for remediation.
4. Reject for this use case.

Minimum requirements:
1. Link your recommendation to model, fairness, mitigation, and monitoring evidence.
2. State controls required for safe operation.
3. Discuss limitations and historical-label risks.
4. Include a concise model card.

Required evidence:
- Deployment decision `[R1]`
- Conditions and controls `[R2]`
- Model card `[R3]`


### Final Deployment Recommendation (Write Here)

Use this structure:
1. Decision: deploy as-is / deploy with controls / remediate / reject.
2. Main evidence supporting the decision.
3. Main fairness concern.
4. Required operational controls.
5. Conditions that would cause the system to be paused or rolled back.
6. What evidence would change your recommendation?


### Model Card (Required)

Fill this template.

- Model name/version:
- Intended use:
- Out-of-scope uses:
- Training data summary:
- Target-label definition and limitations:
- Key features used:
- Protected attributes audited:
- Performance summary:
- Fairness summary:
- Mitigation summary:
- Monitoring commitments:
- Known limitations:
- Human oversight requirements:
- Approval conditions:


## Artifact Export

Export all key outputs (and/or artifacts that support your claim) needed for TA grading reproducibility.

Example expected files:
- `artifacts/d0_data_summary.csv`
- `artifacts/m1_model_comparison.csv`
- `artifacts/m1_threshold_table.csv`
- `artifacts/f2_group_fairness.csv`
- `artifacts/f2_intersectional_fairness.csv`
- `artifacts/i1_individual_proxy_summary.csv`
- `artifacts/g1_mitigation_comparison.csv`
- `artifacts/p0_monitoring_table.csv`
- `artifacts/p1_alerts.csv`
- `artifacts/r0_final_decision.json`


In [17]:
# TODO REQUIRED: Export your grading artifacts with stable filenames.

# Your code here


## Oral Defense Readiness (Mandatory)

Prepare concise answers using your own evidence.

1. Why is your target label useful but not neutral ground truth?
2. Which leakage risks did you remove from the model?
3. Which fairness definition mattered most for your deployment decision?
4. Give one example where fairness definitions conflicted.
5. Which mitigation did you prefer, and what trade-off did it create?
6. What monitoring alert would cause you to pause deployment?


## Evidence Reference Map (Required)

Before submission, map every major claim to a concrete output.

Use this format:
- `[D1]`: table/plot name + claim
- `[M1]`: table/plot name + claim
- `[F1]`: table/plot name + claim
- `[I1]`: table/plot name + claim
- `[G1]`: table/plot name + claim
- `[P1]`: table/plot name + claim
- `[R1]`: final recommendation evidence

Claims that cannot be mapped to evidence are not decision-ready.


## Submission Checklist

- [ ] Team configuration is complete.
- [ ] M0-M5 sections are complete.
- [ ] Historical-label limitations are discussed.
- [ ] Train/validation/test split is leakage-safe.
- [ ] Threshold chosen on validation data only.
- [ ] Group and intersectional fairness tables include sample sizes.
- [ ] At least three fairness definition families are compared.
- [ ] Individual fairness and proxy analyses are included.
- [ ] Two mitigation approaches from different categories are compared.
- [ ] Monitoring batches and alert policy are implemented.
- [ ] One incident investigation worksheet is complete.
- [ ] Model card and final deployment recommendation are complete.
- [ ] Required artifacts are exported under `artifacts/`.
- [ ] Evidence reference map is complete.
- [ ] AI/source disclosure is included if applicable.
